In [ ]:
language = 'pt'

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [ ]:
pip install sounddevice scipy

In [ ]:
import sounddevice as sd
from scipy.io.wavfile import write

def record(sec=5, sample_rate=44100):
    print("Ouvindo...\n")
    
    # Grava o áudio do microfone
    audio = sd.rec(
        frames=int(sec * sample_rate),
        samplerate=sample_rate,
        channels=1,
        dtype='int16'
    )
    
    # Aguarda a gravação terminar
    sd.wait()
    
    # Salva o áudio em um arquivo .mp3
    file_name = 'assets/request_audio.mp3'
    write(file_name, sample_rate, audio)
    
    print("Gravação finalizada!\n")
    return file_name

# Grava o áudio por 5 segundos
record_file = record(sec=5)

# 2. Reconhecimento de Fala com Whisper 🧠

In [ ]:
pip install faster-whisper

In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel("small", device="cpu", compute_type="int8")

print("Transcrevendo...")
segments, info = model.transcribe("assets/request_audio.mp3", language="pt")
transcription = " ".join([seg.text for seg in segments])
print(transcription)

# 3. Integração com a API do Groq 💬

In [ ]:
pip install groq python-dotenv

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()  # carrega o .env automaticamente

In [ ]:
from groq import Groq

client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": transcription}
    ]
)

groq_response = response.choices[0].message.content
print(groq_response)

# 4. Sintetizando a Resposta do Groq Como Voz (gTTS) 🔊

In [ ]:
pip install pygame gtts

In [ ]:
from gtts import gTTS
import pygame

gtts_object = gTTS(text=groq_response, lang=language, slow=False)

response_audio = "assets/response_audio.mp3"
gtts_object.save(response_audio)

pygame.mixer.init()
pygame.mixer.music.load(response_audio)
pygame.mixer.music.play()

# Aguarda terminar de reproduzir
while pygame.mixer.music.get_busy():
    pygame.time.Clock().tick(10)